# Module 14 — Advanced Structures Trie UnionFind SegmentTree

## What you will discover

Every cell below runs this module's **real** problem-bank solutions and asserts
their behaviour. Nothing here prints a claim it has not computed.

The assertions are lifted directly from `problems/tests/`, so they cannot drift
from the implementations — if a signature changes, the tests break first.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

The solutions directory goes on `sys.path` relative to this notebook's own
location. Never hard-code an absolute path — `tools/check_links.py` fails the
build on them, because a path with a username in it works on exactly one
machine.

In [ ]:
import sys
import time
from pathlib import Path

import pytest   # some assertions check that an invalid input RAISES
sys.path.insert(0, str(Path.cwd() / "problems" / "solutions"))

from p01_trie_operations import simulate_trie
from p02_union_find import simulate_union_find
from p03_segment_tree import simulate_segment_tree

print("module 14: Advanced Structures Trie UnionFind SegmentTree")
print("problems available:", 8)
for name in ['p01_trie_operations', 'p02_union_find', 'p03_segment_tree', 'p04_word_search_trie', 'p05_accounts_merge', 'p06_range_min_query', 'p07_count_smaller_after', 'p08_implement_prefix_map']:
    print(f"  {name}")

## 1. Baseline — `p01_trie_operations`

The first property, asserted rather than printed. Read the assertions before
running: each one names a specific input class, and most cross-check against an
independent brute force over the same data.

In [ ]:
ops = [("insert", "apple"), ("search", "apple"), ("search", "app"),
       ("starts_with", "app"), ("insert", "app"), ("search", "app")]
assert simulate_trie(ops) == [True, False, True, True]
# Searching an empty trie.
assert simulate_trie([("search", "a"), ("starts_with", "a")]) == [False, False]
# The empty prefix always matches once anything exists.
assert simulate_trie([("insert", "a"), ("starts_with", "")]) == [True]
# A word is its own prefix.
assert simulate_trie([("insert", "abc"), ("starts_with", "abc")]) == [True]
# A longer query than any stored word.
assert simulate_trie([("insert", "ab"), ("search", "abc")]) == [False]
# Repeated inserts are idempotent.
assert simulate_trie([("insert", "x"), ("insert", "x"), ("search", "x")]) == [True]
with pytest.raises(ValueError):
    simulate_trie([("frobnicate", "x")])
# Cross-check against a plain set of words on a long sequence.
import random
random.seed(11)
vocab = [''.join(random.choice('abc') for _ in range(random.randint(1, 4))) for _ in range(60)]
seq, model, expected = [], set(), []
for _ in range(400):
    w = random.choice(vocab)
    r = random.random()
    if r < 0.4:
        seq.append(('insert', w))
        model.add(w)
    elif r < 0.7:
        seq.append(('search', w))
        expected.append(w in model)
    else:
        seq.append(('starts_with', w))
        expected.append(any(m.startswith(w) for m in model))
assert simulate_trie(seq) == expected

print("all assertions held")

## 2. Predict before you run

You union 100,000 elements into one chain and then query connectivity from the deep end 10,000 times. Predict the total work with path compression, and without it. The two answers differ by about five orders of magnitude.

Commit to an answer before executing the next cell. Predicting and being wrong
is what makes the correction stick; reading the output first does not.

In [ ]:
ops = [("connected", 0, 1), ("union", 0, 1), ("connected", 0, 1), ("count", 0, 0)]
assert simulate_union_find(4, ops) == [False, True, 3]
# Transitivity: merging 0-1 and 1-2 connects 0 and 2.
ops = [("union", 0, 1), ("union", 1, 2), ("connected", 0, 2)]
assert simulate_union_find(3, ops) == [True]
# A node is connected to itself.
assert simulate_union_find(1, [("connected", 0, 0), ("count", 0, 0)]) == [True, 1]
# Redundant unions must not change the count.
ops = [("union", 0, 1), ("union", 1, 0), ("union", 0, 1), ("count", 0, 0)]
assert simulate_union_find(3, ops) == [2]
# Merging everything leaves one set.
ops = [("union", i, i + 1) for i in range(9)] + [("count", 0, 0)]
assert simulate_union_find(10, ops) == [1]
with pytest.raises(ValueError):
    simulate_union_find(2, [("frobnicate", 0, 1)])
# Scale: 10**5 unions in a chain, which is where a missing path
# compression turns O(1) into O(n) and this becomes ~10^10 steps.
big = [("union", i, i + 1) for i in range(99_999)]
big += [("connected", 0, 99_999), ("count", 0, 0)]
assert simulate_union_find(100_000, big) == [True, 1]

print("all assertions held")

## 3. Measurement

Claims about complexity are claims about wall-clock behaviour at scale, so they
have to be measured rather than asserted from the shape of the code.

In [ ]:
started = time.perf_counter()

assert simulate_segment_tree([1, 3, 5], [("query", 0, 2), ("update", 1, 2), ("query", 0, 2)]) == [9, 8]
# Single element.
assert simulate_segment_tree([7], [("query", 0, 0)]) == [7]
assert simulate_segment_tree([7], [("update", 0, 3), ("query", 0, 0)]) == [3]
# Sub-ranges and single-element ranges.
nums = [1, 2, 3, 4, 5]
ops = [("query", 0, 4), ("query", 1, 3), ("query", 2, 2), ("query", 0, 0)]
assert simulate_segment_tree(nums, ops) == [15, 9, 3, 1]
# Negative values.
assert simulate_segment_tree([-1, -2, 3], [("query", 0, 2)]) == [0]
with pytest.raises(ValueError):
    simulate_segment_tree([1], [("frobnicate", 0, 0)])
# Cross-check against a naive model over a long random sequence.
import random
random.seed(5)
base = [random.randint(-50, 50) for _ in range(200)]
model = base[:]
seq, expected = [], []
for _ in range(1500):
    if random.random() < 0.4:
        i, v = random.randrange(len(base)), random.randint(-50, 50)
        seq.append(('update', i, v))
        model[i] = v
    else:
        lo = random.randrange(len(base))
        hi = random.randrange(lo, len(base))
        seq.append(('query', lo, hi))
        expected.append(sum(model[lo : hi + 1]))
assert simulate_segment_tree(base, seq) == expected

elapsed = (time.perf_counter() - started) * 1000
print(f"all assertions held in {elapsed:.2f} ms")

## 4. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
import os

# DELIBERATELY BROKEN - two expected values, both wrong. Fix in place.

expected_problem_count = 99      # how many problems does this module ship?
expected_solution_count = 99     # how many reference solutions are on disk?

problem_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems") if f.startswith("p") and f.endswith(".py")
)
solution_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems" / "solutions")
    if f.startswith("p") and f.endswith(".py")
)

assert expected_problem_count == len(problem_files), (
    f"expected {expected_problem_count} problems, found {len(problem_files)}"
)
assert expected_solution_count == len(solution_files), (
    f"expected {expected_solution_count} solutions, found {len(solution_files)}"
)
print("Both match. Every problem has exactly one reference solution.")

## Takeaways

1. A trie answers prefix questions a hash map cannot; union-find cannot delete an edge.
2. Path compression and union by size are not tuning - they are the data structure.
3. Any cached derived value owes an invalidation path.

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`problems/README.md`](problems/README.md) — all 8 problems, with hint ladders
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — planted defects that exit 0
- [Pattern Recognition Guide](../PATTERN_RECOGNITION_GUIDE.md) — attacking an unseen problem